# Tag 10 – 03: Bildgenerierung über die OpenAI API

Dieses Notebook erzeugt ein Bild über die **OpenAI Images API**. Anders als Stable Diffusion aus Notebook 02 läuft das Modell nicht auf der lokalen RTX 3080, sondern auf der API-Infrastruktur. Dadurch ist kein Modell-Download nötig, aber jeder Aufruf kann Kosten verursachen.

Wir verwenden das aktuelle Bildmodell `gpt-image-2`. Der API-Schlüssel bleibt ausschließlich in einer lokalen `.env`-Datei und wird weder im Notebook noch in Git gespeichert.

## Lokal oder API?

| Variante | Rechenort | Einmaliger Aufwand | Laufende Kosten |
| --- | --- | --- | --- |
| Notebook 02: Stable Diffusion | eigene RTX 3080 | Modell herunterladen, etwa 4–5 GB Speicherplatz | Strom, keine API-Kosten |
| Dieses Notebook: OpenAI API | OpenAI-Infrastruktur | API-Schlüssel einrichten | Kosten pro Bild/API-Aufruf |

Die API ist ein praktisches Beispiel für *Inference as a Service*: Prompt senden, Bilddaten zurückerhalten. Die Architektur des Bildmodells muss dabei nicht lokal geladen oder verwaltet werden.

## Einmalig: `.env` anlegen

Lege im **Projektwurzelordner** – neben `requirements.txt` – eine Datei namens `.env` an. Sie ist bereits in `.gitignore` und darf keinen echten Schlüssel in einem Git-Commit enthalten.

```text
API_TOKEN=xxxxx
```

Ersetze `xxxxx` lokal durch deinen OpenAI-API-Schlüssel. Die bereitgestellte Datei `.env.example` ist nur eine sichere Vorlage. Der Variablenname `API_TOKEN` ist für dieses Kursbeispiel bewusst gewählt; er wird unten explizit an den OpenAI-Client übergeben.

**Wichtig:** Das Ausführen der Generierungszelle schickt den Prompt an die API und kann Kosten verursachen.

In [ ]:
import base64
import os
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image

# Funktioniert sowohl bei Jupyter im Projektwurzelordner als auch bei direkt geöffnetem Notebook.
COURSE_ROOT = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                    if (folder / 'requirements.txt').exists()), Path.cwd())
DAY_DIR = COURSE_ROOT / 'notebooks' / 'Day_10'
ENV_PATH = COURSE_ROOT / '.env'

if not ENV_PATH.exists():
    raise FileNotFoundError(
        f'{ENV_PATH} fehlt. Kopiere .env.example zu .env und trage API_TOKEN ein.'
    )

load_dotenv(ENV_PATH)
api_token = os.environ.get('API_TOKEN')
if not api_token or api_token == 'xxxxx':
    raise ValueError('API_TOKEN fehlt oder enthält noch den Platzhalter. Prüfe die lokale .env-Datei.')

# Der Schlüssel wird nie ausgegeben.
client = OpenAI(api_key=api_token)
print('API-Schlüssel aus .env geladen. Der Schlüssel wird nicht angezeigt.')

## Prompt formulieren

Ein guter Bildprompt benennt Motiv, Umgebung, Bildstil, Licht und wichtige Einschränkungen. Starte mit einer einzelnen Generation, damit die Kosten und das Ergebnis überschaubar bleiben.

In [ ]:
PROMPT = (
    'A friendly red fox reading a book in a small cozy library, '
    'warm evening light, detailed digital illustration, no text in the image'
)
MODEL = 'gpt-image-2'
SIZE = '1024x1024'

print('Modell:', MODEL)
print('Größe:', SIZE)
print('Prompt:', PROMPT)

## API aufrufen und Bild speichern

`client.images.generate(...)` sendet den Prompt an die Images API. Die Antwort enthält die Bilddaten Base64-kodiert. Wir dekodieren sie direkt im Speicher, zeigen das Bild an und speichern es anschließend als PNG.

In [ ]:
# Achtung: Diese Zelle löst einen kostenpflichtigen API-Aufruf aus.
response = client.images.generate(
    model=MODEL,
    prompt=PROMPT,
    size=SIZE,
)

image_bytes = base64.b64decode(response.data[0].b64_json)
image = Image.open(BytesIO(image_bytes))

output_dir = DAY_DIR / 'generated_images'
output_dir.mkdir(exist_ok=True)
output_path = output_dir / 'openai_fox.png'
image.save(output_path)

plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.title(f'{MODEL}: API-generiertes Bild')
plt.axis('off')
plt.show()
print('Gespeichert:', output_path.resolve())

## Experimentieren und Sicherheit

- Verändere nur einen Promptbestandteil pro Durchlauf, beispielsweise Licht oder Stil. So wird sichtbar, welche Instruktion welche Wirkung hatte.
- Erzeuge nicht unkontrolliert viele Bilder in einer Schleife: Jeder Aufruf verursacht API-Nutzung.
- Bei einem Authentifizierungsfehler prüfe zuerst, ob `.env` im Projektwurzelordner liegt und `API_TOKEN` richtig gesetzt ist.
- Bei einem Berechtigungs- oder Abrechnungsfehler ist der Schlüssel zwar lesbar, aber das API-Konto hat möglicherweise keinen Zugriff oder kein aktiviertes Billing.
- Teile niemals `.env`, API-Keys oder Notebook-Ausgaben, in denen ein Schlüssel versehentlich erscheint. Sollte ein Schlüssel öffentlich geworden sein, widerrufe ihn im API-Dashboard und erstelle einen neuen.

## Zusammenfassung

- `python-dotenv` lädt den geheimen Schlüssel lokal aus `.env`; der Python-Code enthält keinen Schlüssel.
- Der OpenAI-Client ruft die Images API auf und liefert Base64-kodierte Bilddaten zurück.
- Das Bild wird lokal gespeichert, das Modell selbst aber nicht auf der eigenen GPU ausgeführt.
- Das lokale Diffusionsbeispiel und die API-Variante zeigen zwei unterschiedliche Wege zur Bildgenerierung: eigene Hardware oder externe Modell-Inferenz.